In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cpu


In [25]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [26]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])


tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [27]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)


inputs:
tensor([[68, 71, 68, 73, 61, 78,  1, 72],
        [ 1, 55, 78,  1, 59, 65, 78, 62],
        [76, 61, 58, 71, 58,  1, 73, 61],
        [67, 57,  1, 72, 68, 68, 67,  1]])
targets:
tensor([[71, 68, 73, 61, 78,  1, 72, 69],
        [55, 78,  1, 59, 65, 78, 62, 67],
        [61, 58, 71, 58,  1, 73, 61, 58],
        [57,  1, 72, 68, 68, 67,  1, 73]])


In [28]:
x = train_data[:block_size]
y = train_data[:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, "target is", target)

when input is tensor([80]) target is tensor(80)
when input is tensor([80,  1]) target is tensor(1)
when input is tensor([80,  1,  1]) target is tensor(1)
when input is tensor([80,  1,  1, 28]) target is tensor(28)
when input is tensor([80,  1,  1, 28, 39]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42]) target is tensor(42)
when input is tensor([80,  1,  1, 28, 39, 42, 39]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42, 39, 44]) target is tensor(44)


In [29]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [30]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=-1) 
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1) 
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



Uzt,e,)wmP[!kUOFa3[uWWlyPj!?KGL6[v(4Gd2t]B?qta lvI9HiEZxNfoWj7n;)18Dh1W;ZF1]
SWjGy-m?wD'ztQTR)eupL-?iJ
﻿E&9d2;k:-IpoAflM
s4gRv,28A*R')zXbf:LvtgeBn:iVQ
y;91TjF[s[6SW4Nl6oA !a"XP0rZFX&XOCf"NJAa0dIhg94hB"7ZruODgZ;ZH3mGd!e8S_;Lv;.tjbg*WeZNS)M2PRwg(&oc".iM_ce﻿C2R)OaVWa8jGNm&D*5swZhkZI&y
aLU
M8
y61DUN9aLMjg9[x,6Su48i'4tjbIemhnc !t[03K.'fq*)OCfliOC0l1UZE5N;) Xnpl.BrAhIx-628FHP[A]
l
p-nlts?T9;55? e''xguM6i7.a&F6qQ
[VAf4(&aBQzl_;_o]xL﻿j)Oh;_
fxFNw,7vIS4rJ5uFL2(M&*2i5chk*a1U:AMw.U_e7QC2PnSj;aCB5*_D4t[dIMs


In [31]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.880, val loss: 4.913
step: 250, train loss: 4.822, val loss: 4.840
step: 500, train loss: 4.760, val loss: 4.773
step: 750, train loss: 4.684, val loss: 4.695
4.904557228088379


In [32]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


 j2?;wJWmom*Sq'55cQzl2dE﻿.'wlVFfeT(KfOjq_uf:YyiHE!:bxfB9S*Z
!wgy0!Tp BU?J5P0]Xn!aDMB OW".'ll:t(2F[K&Kj1u2Y)W
OYyC8EN!ztghcbx0u&drBWuKhmZ9d[)X!?.aBQn!EbvIWB﻿GR[AM
Wrx*d3Ug-7e*5*h;Znp*r2(C([B&CBs

s!G:S_u2T;p_1EpU?PcSZ*y;aguD"O8Rv9h]anfCv:pLF:r.xmHBy*h
x﻿om&ltG,dB9v6ru&aB9qEy;dNt.D1LpixW﻿1.?SpkZn0(4M
rouD*55
y
r97nwc(kvcCZx,﻿:KEZng:Kla,64BW
rZIgNq[_7NVpCP(3Df?qDArlUwp)Mwsx?t_DXJKQfeBvpX_P
6wl:J﻿m!0Q?o-U?-IWr﻿T1t
0yrJ8),_(Qsh8[cum_a7:K9d[_Pq?Jio?V63BHv:WQpN9UA
s4(tAhJ!_Hi6(4 Cc
0bki6.-6rOu(cSGd[."l
